In [4]:
!pip install langchain chromadb openai tiktoken pypdf langchain_community huggingface-hub langchain_huggingface

INFO: pip is looking at multiple versions of langchain-huggingface to determine which version is compatible with other requirements. This could take a while.


In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-MiniLM-L6-v2')

In [7]:
from langchain.vectorstores import Chroma

In [12]:
from langchain.schema import Document

doc1 = Document(
    page_content = "Machine Learning helps computers learn from data using algorithms such as Linear Regression, SVM, and Neural Networks. It involves data preprocessing, model training, and evaluation.",
    metadata = {
        "topic": "Machine Learning",
        "source": "internal_notes",
        "doc_id": "ml_001",
        "author": "Angel"
    }
)

doc2 = Document(
    page_content = "Remote Work Policy: Employees may work from home for two days a week. Core working hours are from 10 AM to 4 PM. Prior approval from the manager is required.",
    metadata = {
        "topic": "HR Policy",
        "source": "company_handbook",
        "doc_id": "hr_2025",
        "version": "1.2"
    }
)

doc3 = Document(
    page_content = "Chicken Momos Recipe: Mix minced chicken, onion, garlic, and spices. Fill wrappers, fold into dumplings, and steam for 15 minutes. Serve with spicy chutney.",
    metadata = {
        "topic": "Recipe",
        "cuisine": "Nepalese",
        "doc_id": "recipe_007",
        "difficulty": "easy"
    }
)

doc4 = Document(
    page_content = "Networking: To subnet 192.168.1.0/24 into four equal subnets, use a /26 mask. Subnets: 192.168.1.0/26, .64/26, .128/26, .192/26.",
    metadata = {
        "topic": "Networking",
        "doc_id": "net_004",
        "chapter": "Subnetting Basics"
    }
)

doc5 = Document(
    page_content = "Efficient Transformer Models: Techniques like sparse attention, linear attention, and low-rank approximations reduce computational cost for long sequences.",
    metadata = {
        "topic": "NLP Research",
        "doc_id": "nlp_trf_2025",
        "papers": "Longformer, Linformer, Reformer"
    }
)


In [13]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [14]:
vector_store = Chroma(
    embedding_function= embeddings,
    persist_directory= 'chroma_db',
    collection_name = 'sample'
)

In [15]:
# adding documents

vector_store.add_documents(docs)

['ed607901-f018-4114-aa7d-18e11b2edc76',
 '15997450-b865-44d8-983b-7b2e85038783',
 '9c3102ca-9837-4db1-8941-941b301e6283',
 'd54cb692-35fa-4007-84dd-b633fffdcea1',
 'cb5a6e00-b6e7-4831-95d0-f4adb3cfc973']

In [18]:
vector_store.get(include = ['embeddings', 'documents', 'metadatas'])

{'ids': ['ed607901-f018-4114-aa7d-18e11b2edc76',
  '15997450-b865-44d8-983b-7b2e85038783',
  '9c3102ca-9837-4db1-8941-941b301e6283',
  'd54cb692-35fa-4007-84dd-b633fffdcea1',
  'cb5a6e00-b6e7-4831-95d0-f4adb3cfc973'],
 'embeddings': array([[-0.02627911,  0.00823283,  0.0359384 , ...,  0.06104868,
          0.05946112, -0.06594927],
        [ 0.02255765, -0.02570075,  0.05674962, ...,  0.01553026,
         -0.08846238,  0.03759841],
        [-0.05792968, -0.09166465, -0.01407034, ...,  0.0659272 ,
          0.00724542,  0.02797731],
        [ 0.08851767, -0.0057851 , -0.06392051, ...,  0.05948455,
         -0.05842233, -0.03267663],
        [-0.05550006, -0.04156221,  0.01346233, ...,  0.02650432,
         -0.06956155,  0.01886295]]),
 'documents': ['Machine Learning helps computers learn from data using algorithms such as Linear Regression, SVM, and Neural Networks. It involves data preprocessing, model training, and evaluation.',
  'Remote Work Policy: Employees may work from home for

In [20]:
vector_store.similarity_search(
    query = "How do machines learn from data?",
    k=1
)

[Document(metadata={'source': 'internal_notes', 'topic': 'Machine Learning', 'author': 'Angel', 'doc_id': 'ml_001'}, page_content='Machine Learning helps computers learn from data using algorithms such as Linear Regression, SVM, and Neural Networks. It involves data preprocessing, model training, and evaluation.')]

In [23]:
vector_store.similarity_search_with_score(
    query = "How do machines learn from data?",
    k=1
)

[(Document(metadata={'author': 'Angel', 'topic': 'Machine Learning', 'doc_id': 'ml_001', 'source': 'internal_notes'}, page_content='Machine Learning helps computers learn from data using algorithms such as Linear Regression, SVM, and Neural Networks. It involves data preprocessing, model training, and evaluation.'),
  0.7175586223602295)]

In [30]:
# meta data filtering
vector_store.similarity_search(
    query = " ",
    filter = {"topic": "NLP Research"}
)

[Document(metadata={'topic': 'NLP Research', 'doc_id': 'nlp_trf_2025', 'papers': 'Longformer, Linformer, Reformer'}, page_content='Efficient Transformer Models: Techniques like sparse attention, linear attention, and low-rank approximations reduce computational cost for long sequences.')]

In [33]:
# update documents

updated_docs1 = Document(
    page_content = (
        "Machine Learning enables computers to learn patterns from data and make predictions "
        "or decisions without being explicitly programmed. It includes key processes such as "
        "data collection, preprocessing, feature engineering, model training, hyperparameter "
        "tuning, and evaluation. Common algorithms include Linear Regression, Support Vector "
        "Machines (SVM), Decision Trees, and Deep Neural Networks. ML is widely used in "
        "applications like recommendation systems, fraud detection, image recognition, and "
        "natural language processing."
    ),
    metadata = {
        "topic": "Machine Learning",
        "source": "internal_notes",
        "doc_id": "ml_001",
        "author": "Angel"
    }
)


vector_store.update_documents(
    ids=["ed607901-f018-4114-aa7d-18e11b2edc76"],
    documents=[updated_docs1]
)

In [34]:
vector_store.get(include = ['embeddings', 'documents', 'metadatas'])

{'ids': ['ed607901-f018-4114-aa7d-18e11b2edc76',
  '15997450-b865-44d8-983b-7b2e85038783',
  '9c3102ca-9837-4db1-8941-941b301e6283',
  'd54cb692-35fa-4007-84dd-b633fffdcea1',
  'cb5a6e00-b6e7-4831-95d0-f4adb3cfc973'],
 'embeddings': array([[-0.08148916, -0.01626199,  0.02967683, ..., -0.00379881,
          0.0572073 , -0.02646319],
        [ 0.02255765, -0.02570075,  0.05674962, ...,  0.01553026,
         -0.08846238,  0.03759841],
        [-0.05792968, -0.09166465, -0.01407034, ...,  0.0659272 ,
          0.00724542,  0.02797731],
        [ 0.08851767, -0.0057851 , -0.06392051, ...,  0.05948455,
         -0.05842233, -0.03267663],
        [-0.05550006, -0.04156221,  0.01346233, ...,  0.02650432,
         -0.06956155,  0.01886295]]),
 'documents': ['Machine Learning enables computers to learn patterns from data and make predictions or decisions without being explicitly programmed. It includes key processes such as data collection, preprocessing, feature engineering, model training, hyp

In [35]:
vector_store.delete(ids = ['ed607901-f018-4114-aa7d-18e11b2edc76'])

In [36]:
vector_store.get(include = ['embeddings', 'documents', 'metadatas'])

{'ids': ['15997450-b865-44d8-983b-7b2e85038783',
  '9c3102ca-9837-4db1-8941-941b301e6283',
  'd54cb692-35fa-4007-84dd-b633fffdcea1',
  'cb5a6e00-b6e7-4831-95d0-f4adb3cfc973'],
 'embeddings': array([[ 0.02255765, -0.02570075,  0.05674962, ...,  0.01553026,
         -0.08846238,  0.03759841],
        [-0.05792968, -0.09166465, -0.01407034, ...,  0.0659272 ,
          0.00724542,  0.02797731],
        [ 0.08851767, -0.0057851 , -0.06392051, ...,  0.05948455,
         -0.05842233, -0.03267663],
        [-0.05550006, -0.04156221,  0.01346233, ...,  0.02650432,
         -0.06956155,  0.01886295]]),
 'documents': ['Remote Work Policy: Employees may work from home for two days a week. Core working hours are from 10 AM to 4 PM. Prior approval from the manager is required.',
  'Chicken Momos Recipe: Mix minced chicken, onion, garlic, and spices. Fill wrappers, fold into dumplings, and steam for 15 minutes. Serve with spicy chutney.',
  'Networking: To subnet 192.168.1.0/24 into four equal subnet